**Este notebook apresenta a exploração visual dos dados do inventário agrícola da Atvos, com foco em estatística descritiva e relacionamentos entre variáveis.**

Os dados utilizados foram pré-processados pelo pipeline de limpeza `limpeza.py`, gerando o arquivo `inventario_silver.csv` com registros validados e padronizados.

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import textwrap

In [ ]:
warnings.filterwarnings('ignore')

# Lê o CSV limpo 
df = pd.read_csv('../../../data/inventario_silver.csv')

# Converte colunas numéricas
for col in ['area_ha', 'tch_prod', 'area_prod', 'ton_estim']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Paleta e estilo global 
CORES = ['#1B4332','#2D6A4F','#40916C','#74C69D','#B7E4C7','#95D5B2','#52B788','#081C15']
BG    = '#F8FAF9'
DARK  = '#1B4332'

plt.rcParams.update({
    'figure.facecolor': BG,
    'axes.facecolor':   BG,
    'axes.edgecolor':   '#C8DDD4',
    'axes.labelcolor':  DARK,
    'xtick.color':      DARK,
    'ytick.color':      DARK,
    'text.color':       DARK,
    'grid.color':       '#C8DDD4',
    'grid.linestyle':   '--',
    'grid.alpha':       0.6,
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})

print(f'Dados carregados: {len(df)} registros, {df.shape[1]} colunas')
print('\nEstatística descritiva — colunas numéricas principais:')
df[['area_ha', 'tch_prod', 'area_prod', 'ton_estim']].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6), facecolor=BG)

unidades  = sorted(df['unidade_industrial'].dropna().unique())
dados_box = [df[df['unidade_industrial'] == u]['area_ha'].dropna().values for u in unidades]

bp = ax.boxplot(dados_box, patch_artist=True,
                medianprops=dict(color='white', linewidth=2.5),
                whiskerprops=dict(color=DARK, linewidth=1.2),
                capprops=dict(color=DARK, linewidth=1.5),
                flierprops=dict(marker='o', color='#40916C', alpha=0.4, markersize=4))

for patch, cor in zip(bp['boxes'], CORES):
    patch.set_facecolor(cor)
    patch.set_alpha(0.85)

ax.set_xticks(range(1, len(unidades) + 1))
ax.set_xticklabels(unidades, rotation=20, ha='right', fontsize=10)
ax.set_title('Distribuição de Área (ha) por Unidade Industrial', fontweight='bold', pad=12)
ax.set_ylabel('Área (ha)')
ax.set_xlabel('Unidade Industrial')
ax.yaxis.grid(True)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('../../../data/grafico1_area_por_unidade.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico 1 salvo em data/grafico1_area_por_unidade.png')

In [ ]:
fig_height = max(10, len(contagem_solo) * 0.45)
fig, ax = plt.subplots(figsize=(16, fig_height), facecolor=BG)

# Quebra labels longos em múltiplas linhas ao invés de truncar
labels_wrapped = ['\n'.join(textwrap.wrap(l, width=40)) for l in contagem_solo.index]

bars = ax.barh(labels_wrapped, contagem_solo.values,
               color=[CORES[i % len(CORES)] for i in range(len(contagem_solo))],
               edgecolor='none', height=0.6)

for bar, val in zip(bars, contagem_solo.values):
    ax.text(val + contagem_solo.max() * 0.01,
            bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=9, color=DARK, fontweight='bold')

ax.set_title('Quantidade de Talhões por Tipo de Solo', fontweight='bold', pad=14, fontsize=14)
ax.set_xlabel('Nº de Talhões', fontsize=11)
ax.xaxis.grid(True)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='y', labelsize=8.5)

# Margem esquerda generosa para os labels não cortarem
plt.subplots_adjust(left=0.45)
plt.savefig('../../../data/grafico2_talhoes_por_solo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico 2 salvo em data/grafico2_talhoes_por_solo.png')

In [ ]:
# Filtra apenas registros com tch_prod válido
df_tch = df[df['tch_prod'].notna() & df['tipo_solo'].notna() & df['unidade_industrial'].notna()]

# Monta pivot completo
pivot = (df_tch.groupby(['unidade_industrial', 'tipo_solo'])['tch_prod']
               .mean()
               .unstack(fill_value=0))

# Divide tipos de solo em 2 grupos pela frequência de ocorrência
contagem = df_tch['tipo_solo'].value_counts()
mediana  = contagem.median()
solos_a  = contagem[contagem >= mediana].index.tolist()   # mais frequentes
solos_b  = contagem[contagem <  mediana].index.tolist()   # menos frequentes

pivot_a = pivot[[c for c in pivot.columns if c in solos_a]]
pivot_b = pivot[[c for c in pivot.columns if c in solos_b]]

vmax = pivot.values.max()

def encurtar(labels, n=30):
    return [l[:n] + '...' if len(l) > n else l for l in labels]

def plot_heatmap(ax, pivot_df, titulo, vmax):
    im = ax.imshow(pivot_df.values, aspect='auto', cmap='YlGn', vmin=0, vmax=vmax)
    ax.set_xticks(range(len(pivot_df.columns)))
    ax.set_xticklabels(encurtar(list(pivot_df.columns)), rotation=35, ha='right', fontsize=7.5)
    ax.set_yticks(range(len(pivot_df.index)))
    ax.set_yticklabels(pivot_df.index, fontsize=9)
    ax.set_title(titulo, fontweight='bold', pad=10, fontsize=12)
    for i in range(pivot_df.shape[0]):
        for j in range(pivot_df.shape[1]):
            val = pivot_df.values[i, j]
            if val > 0:
                ax.text(j, i, f'{val:.0f}', ha='center', va='center',
                        fontsize=7, color=DARK, fontweight='bold')
    return im

# Grupo A: solos mais frequentes
fig, ax = plt.subplots(figsize=(16, 7), facecolor=BG)
im = plot_heatmap(ax, pivot_a,
                  f'TCH Médio (ton/ha) — Grupo A: Tipos de Solo Mais Frequentes ({len(solos_a)} tipos)', vmax)
plt.colorbar(im, ax=ax, label='TCH Médio (ton/ha)', shrink=0.8)
plt.tight_layout()
plt.savefig('../../../data/grafico3a_heatmap_grupoA.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Grupo A salvo — {len(solos_a)} tipos de solo')

# Grupo B: solos menos frequentes
fig, ax = plt.subplots(figsize=(16, 7), facecolor=BG)
im = plot_heatmap(ax, pivot_b,
                  f'TCH Médio (ton/ha) — Grupo B: Tipos de Solo Menos Frequentes ({len(solos_b)} tipos)', vmax)
plt.colorbar(im, ax=ax, label='TCH Médio (ton/ha)', shrink=0.8)
plt.tight_layout()
plt.savefig('../../../data/grafico3b_heatmap_grupoB.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Grupo B salvo — {len(solos_b)} tipos de solo')
print('\nTodos os gráficos salvos na pasta data/')